In [0]:
%sql
-- Run these in the SQL editor (not in the pipeline) after the Gold run.

-- 1. Grain check: the fact table keeps one row per Silver transaction.
--    The join to dim_customer must not duplicate or drop rows.
SELECT
  (SELECT COUNT(*) FROM dev_kyc.silver.transactions)                 AS silver_rows,
  (SELECT COUNT(*) FROM dev_kyc.gold.fact_transactions)              AS fact_rows,
  (SELECT COUNT(*) - COUNT(DISTINCT txn_id) FROM dev_kyc.gold.fact_transactions) AS duplicate_txn_ids;

In [0]:
%sql
-- 2. Dimension sizes.
SELECT 'dim_customer' AS object, COUNT(*) AS row_count FROM dev_kyc.gold.dim_customer
UNION ALL SELECT 'dim_date',      COUNT(*) FROM dev_kyc.gold.dim_date
UNION ALL SELECT 'dim_merchant',  COUNT(*) FROM dev_kyc.gold.dim_merchant
UNION ALL SELECT 'watchlist_matches',   COUNT(*) FROM dev_kyc.gold.watchlist_matches
UNION ALL SELECT 'customer_risk_daily', COUNT(*) FROM dev_kyc.gold.customer_risk_daily;

In [0]:
%sql
-- 3. Exactly one current version per customer (should return no rows).
SELECT customer_id, COUNT(*) AS current_versions
FROM dev_kyc.gold.dim_customer
WHERE is_current
GROUP BY customer_id
HAVING COUNT(*) > 1;

In [0]:
%sql
-- 4. Orphans: transactions whose customer has no record yet.
--    Expect a few if the late file references customers from days you have not loaded.
--    They resolve after you load the remaining customer days and refresh.
SELECT customer_match_status, COUNT(*) AS txns
FROM dev_kyc.gold.fact_transactions
GROUP BY customer_match_status;

In [0]:
%sql
-- 5. Point-in-time join in action: the same customer maps to different
--    dimension rows (and attribute values) before and after a change.
SELECT c.customer_id, f.txn_id, f.txn_ts, c.risk_rating, c.city, c.version_start_at, c.is_current
FROM dev_kyc.gold.fact_transactions f
JOIN dev_kyc.gold.dim_customer c ON f.customer_sk = c.customer_sk
WHERE c.customer_id IN (
  SELECT customer_id FROM dev_kyc.gold.dim_customer GROUP BY customer_id HAVING COUNT(*) > 1
)
ORDER BY c.customer_id, f.txn_ts
LIMIT 100;

In [0]:
%sql
-- 6. Structuring alerts (expect about 3 customers per loaded day).
SELECT txn_date, customer_id, near_threshold_count, total_amount
FROM dev_kyc.gold.customer_risk_daily
WHERE structuring_alert
ORDER BY txn_date, near_threshold_count DESC;

In [0]:
%sql
-- 7. High-risk geography alerts (expect about 4 per loaded day).
SELECT txn_date, customer_id, high_risk_country_count, max_amount
FROM dev_kyc.gold.customer_risk_daily
WHERE high_risk_geo_alert
ORDER BY txn_date;


In [0]:
%sql
-- 8. Potential watchlist matches. Customers sharing a name with a listed
--    entry show up too: these are the false positives analysts would clear.
SELECT m.customer_id, c.first_name, c.last_name, m.watchlist_id, w.full_name,
       m.match_type, m.country_match, m.match_confidence, m.review_status
FROM dev_kyc.gold.watchlist_matches m
JOIN dev_kyc.gold.dim_customer c ON m.customer_id = c.customer_id AND c.is_current
JOIN dev_kyc.silver.watchlist w  ON m.watchlist_id = w.watchlist_id
ORDER BY m.watchlist_id, m.customer_id;

In [0]:
%sql
-- 9. A dashboard-style query: top customers by alert days.
SELECT customer_id,
       SUM(CASE WHEN structuring_alert THEN 1 ELSE 0 END)  AS structuring_days,
       SUM(CASE WHEN high_risk_geo_alert THEN 1 ELSE 0 END) AS high_risk_geo_days,
       SUM(total_amount) AS total_amount
FROM dev_kyc.gold.customer_risk_daily
GROUP BY customer_id
HAVING structuring_days > 0 OR high_risk_geo_days > 0
ORDER BY structuring_days DESC, high_risk_geo_days DESC
LIMIT 20;
